example_vocab

In [ ]:
vocab = {
    0: b' ', 1: b'a', 2:
b'c', 3: b'e', 4: b'h', 5: b't', 6: b'th', 7: b' c', 8: b' a', 9: b'the', 10: b' at'}

In [ ]:
print(vocab)

example_merges

In [ ]:
merges = [(b't', b'h'), (b' ', b'c'), (b' ', b'a'), (b'th', b'e'),
(b' a', b't')]

In [ ]:
print(merges)

the tokenizer must encoded the text "the cat ate" into these sequence

In [ ]:
true_tokens_encoded = [9, 7, 1, 5, 10, 3]

# start example

In [ ]:
from cs336_basics.bpe_train.utils import find_chunk_boundaries
from cs336_basics.bpe_train.utils import convert_key_to_tuple_of_bytes
import regex as re

TOKENIZE_PATTERN=r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
TOKENIZE_PATTERN = re.compile(TOKENIZE_PATTERN)

In [ ]:
def split_on_special_tokens(chunk: str, special_tokens: list[str]) -> list[str]:
    """
    Split the chunk on special tokens and return a list of sub-chunks.
    """
    # Create a regex pattern to match any of the special tokens
    special_token_pattern = "|".join(re.escape(token) for token in special_tokens)
    split_pattern = re.compile(special_token_pattern)

    # Split the chunk on the special tokens
    sub_chunks = split_pattern.split(chunk)

    # Filter out empty strings and return the list of sub-chunks
    return [sub_chunk for sub_chunk in sub_chunks if sub_chunk.strip()]

In [ ]:
def pre_tokenize_chunk(text,special_tokens):
    """
    Pre-tokenize a text in chunks.

    Args:
        text (str): The text chunk to pre-tokenize.
        special_tokens (list[str]): List of special tokens to split on.

    Returns:
        int: Number of pre-tokens in the chunk.
    """
    
    chunk_without_special_tokens = split_on_special_tokens(text, special_tokens)
    text_chunks = []
    for sub_chunk in chunk_without_special_tokens:
        text_chunks.extend([t.group() for t in re.finditer(TOKENIZE_PATTERN, sub_chunk)])
    return text_chunks

In [ ]:
text = 'the cat ate'

split the text in the unique elements following the regex pattern

In [ ]:
pre_tokens = pre_tokenize_chunk(text, ["<|endoftext|>"])

In [ ]:
print(pre_tokens)

In [ ]:
def _merge_pair_in_token(pair, token):
    first, second = pair
    merged_token = []
    i = 0
    while i < len(token):
        if i < len(token) - 1 and token[i] == first and token[i + 1] == second:
            merged_token.append(first + second)
            i += 2
        else:
            merged_token.append(token[i])
            i += 1
    return tuple(merged_token)

In [ ]:
def _get_pair_from_token_v2(token):
    pair_list = []
    #for i in range(len(token) - 1):
    #    pair_list.append((token[i], token[i + 1]))
    for t1, t2 in zip(token, token[1:]):
        pair_list.append((t1, t2))
    return pair_list

In [ ]:
pre_tokens

convert the text into tuples of bytes

In [ ]:
pre_tokens_bytes = [convert_key_to_tuple_of_bytes(pre_tok) for pre_tok in pre_tokens]

In [ ]:
pre_tokens_bytes

In [ ]:
# loop through merges and apply them to the pre-tokenized bytes
"""
do_loop = False
if do_loop:
    no_merges_applied = 0
    merges_applied = 0
    start_pre_tok_bytes = pre_tok_bytes
    for m in merges:
        new_pre_tok_bytes = _merge_pair_in_token(m, start_pre_tok_bytes)
        if new_pre_tok_bytes == start_pre_tok_bytes:
            print(f"No merge applied for pair {m}")
            no_merges_applied += 1
        else:
            print(f"Merge applied for pair {m}")
            merges_applied += 1
        start_pre_tok_bytes = new_pre_tok_bytes
    start_pre_tok_bytes
"""

map each merge to an integer

In [ ]:
merges_dict = {m: i for i, m in enumerate(merges)}

merging loop: <br>
- get from each pre token the pairs
- check if there are some pair that needs to be merged
- merged that pair inside the pre token

In [ ]:
tokens_merged = []
for pre_tok_bytes in pre_tokens_bytes:

    print("pre-tokenized bytes:", pre_tok_bytes)
    tokens_pair = _get_pair_from_token_v2(pre_tok_bytes)

    # get the first pair to merge based on the merges dictionary
    pair_to_merge = min(tokens_pair, key=lambda x: merges_dict.get(x, float('inf')))
    
    while merges_dict.get(pair_to_merge) is not None:
        print("merging", pair_to_merge, "in", pre_tok_bytes)
        pre_tok_bytes = _merge_pair_in_token(pair_to_merge, pre_tok_bytes)
        tokens_pair = _get_pair_from_token_v2(pre_tok_bytes)
        if not tokens_pair:
            break
        pair_to_merge = min(tokens_pair, key=lambda x: merges_dict.get(x, float('inf')))
    tokens_merged.append(pre_tok_bytes)
    print("-"*20)
    print("\n")
    


In [ ]:
print(f"Final tokens: {tokens_merged}")

# map tokens to dict

In [ ]:
vocab

In [ ]:
inv_vocab = {v: k for k, v in vocab.items()}

In [ ]:
inv_vocab

In [ ]:
# slow method 
tokens_encoded = []
for token_tuple in tokens_merged:
    for token in token_tuple:
        tokens_encoded.append(inv_vocab[token])


In [ ]:
tokens_encoded_v2 = [inv_vocab[t] for token_tuple in tokens_merged for t in token_tuple]

In [ ]:
tokens_encoded_v2

# everything in a single loop

In [ ]:
tokens_encoded = []
for pre_tok_bytes in pre_tokens_bytes:

    print("pre-tokenized bytes:", pre_tok_bytes)
    tokens_pair = _get_pair_from_token_v2(pre_tok_bytes)

    # get the first pair to merge based on the merges dictionary
    pair_to_merge = min(tokens_pair, key=lambda x: merges_dict.get(x, float('inf')))
    
    while merges_dict.get(pair_to_merge) is not None:
        print("merging", pair_to_merge, "in", pre_tok_bytes)
        pre_tok_bytes = _merge_pair_in_token(pair_to_merge, pre_tok_bytes)
        tokens_pair = _get_pair_from_token_v2(pre_tok_bytes)
        if not tokens_pair:
            break
        pair_to_merge = min(tokens_pair, key=lambda x: merges_dict.get(x, float('inf')))

    tokens_ids = [inv_vocab[token] for token in pre_tok_bytes]
    tokens_encoded.extend(tokens_ids)
    print("-"*20)
    print("\n")
    


In [ ]:
tokens_encoded

# decoding

In [ ]:
tokens_decoded = [vocab[token_id] for token_id in tokens_encoded]
tokens_decoded


In [ ]:
vocab

In [ ]:
def decode_id(id, vocab):
    byte_seq = vocab[id]
    txt = byte_seq.decode('utf-8', errors='replace')
    return txt


In [ ]:
original_text = "".join([decode_id(token_id, vocab) for token_id in tokens_encoded])
original_text